# Indian CGD Gas Pipeline IDS — ML Pipeline
## Phase 7: Dataset Analysis + IDS Model Training

**Regulatory basis:** PNGRB T4S (2024) — 14–26 barg steel grid  
**Gas SG:** 0.57 (GAIL/ONGC supply)  
**Network:** 20 nodes, 20 edges + 2 resilience edges (E21, E22)  
**Dataset:** ~774,000 rows, 340 baseline + 90 attack scenarios  

---
### Notebook Sections
1. Dataset Loading & Validation
2. Exploratory Data Analysis (EDA)
3. Indian CGD Pressure Range Verification
4. Feature Engineering (Weymouth Residuals, Kirchhoff Balance)
5. Class Distribution & Imbalance Analysis
6. Unsupervised Anomaly Detection (Isolation Forest + LSTM-AE)
7. Supervised Multi-Class Classification (XGBoost + LSTM)
8. Cross-Topology Validation (Leave-One-Topology-Out)
9. SHAP Feature Attribution
10. Attack-Type Performance Breakdown
11. Resilience Scenario Analysis
12. Export Results for Paper

In [ ]:
# ── 0. Environment Setup ────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
import json, os, glob
from pathlib import Path

# ML libraries
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, f1_score, ConfusionMatrixDisplay
)
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
import shap

# Deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
sns.set_theme(style='whitegrid')

# ── Path configuration ──────────────────────────────────────────────────
DATA_DIR   = Path('automated_dataset')
SCEN_DIR   = DATA_DIR / 'scenarios'
OUTPUT_DIR = Path('ml_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# Indian CGD constants (PNGRB T4S)
P_MIN_BARG  = 14.0   # minimum steel grid delivery
P_MAX_BARG  = 26.0   # MAOP
P_NOM_BARG  = 22.0   # nominal mid-range
PRS1_SP     = 18.0   # DRS1 setpoint
PRS2_SP     = 14.0   # DRS2 setpoint
GAS_SG      = 0.57

print('Environment ready.')
print(f'Data directory: {DATA_DIR.absolute()}')

## 1. Dataset Loading & Validation

In [ ]:
# ── 1.1 Load master dataset ─────────────────────────────────────────────
master_path = DATA_DIR / 'ml_dataset_final.csv'

if master_path.exists():
    df = pd.read_csv(master_path, low_memory=False)
    print(f'Loaded master dataset: {len(df):,} rows × {len(df.columns)} cols')
else:
    # Load and concatenate individual scenario files
    files = sorted(SCEN_DIR.glob('scenario_*.csv'))
    print(f'Loading {len(files)} scenario files...')
    dfs = [pd.read_csv(f, low_memory=False) for f in files]
    df  = pd.concat(dfs, ignore_index=True)
    print(f'Concatenated: {len(df):,} rows × {len(df.columns)} cols')

# Parse column groups
PRESSURE_COLS  = [c for c in df.columns if c.endswith('_bar') and not c.startswith('ekf')]
FLOW_COLS      = [c for c in df.columns if c.endswith('_scmd')]
EKF_P_COLS     = [c for c in df.columns if c.startswith('ekf_p_')]
EKF_RESID_COLS = [c for c in df.columns if c.startswith('ekf_resid_')]
EQUIP_COLS     = ['CS1_ratio','CS1_power_kW','CS2_ratio','CS2_power_kW',
                  'PRS1_throttle','PRS2_throttle','STO_inventory']
CUSUM_COLS     = ['cusum_S_upper','cusum_S_lower','cusum_alarm','cusum_z']
LABEL_COLS     = ['FAULT_ID','ATTACK_ID','ATTACK_NAME','MITRE_ID','label']
SCENARIO_COLS  = ['scenario_id','source_config','demand_profile',
                  'valve_config','cs_mode','storage_init','scenario_resilience_mode']

print(f'\nPressure columns: {len(PRESSURE_COLS)}')
print(f'Flow columns:     {len(FLOW_COLS)}')
print(f'EKF columns:      {len(EKF_P_COLS)} estimates + {len(EKF_RESID_COLS)} residuals')
print(f'Label columns:    {LABEL_COLS}')

In [ ]:
# ── 1.2 Indian CGD Pressure Range Validation ────────────────────────────
print('=== PNGRB T4S Pressure Validation ===')
p_stats = df[PRESSURE_COLS].describe()

# Check all pressures within CGD bounds
out_of_range = {}
for col in PRESSURE_COLS:
    mn = df[col].min()
    mx = df[col].max()
    if mx > 28.0 or mn < 0:
        out_of_range[col] = (mn, mx)

if out_of_range:
    print(f'WARN: {len(out_of_range)} columns out of PNGRB T4S range:')
    for c,(mn,mx) in out_of_range.items():
        print(f'  {c}: min={mn:.2f}, max={mx:.2f} barg')
else:
    print(f'✓ All {len(PRESSURE_COLS)} pressure columns within 0–28 barg')

print(f'\nNominal pressure summary:')
key_nodes = ['p_S1_bar','p_CS1_bar','p_PRS1_bar','p_D1_bar',
             'p_CS2_bar','p_PRS2_bar','p_D3_bar']
key_nodes = [c for c in key_nodes if c in df.columns]
print(df[key_nodes].describe().round(3).to_string())

# Verify PRS setpoints (PNGRB T4S: PRS1=18 barg, PRS2=14 barg)
if 'p_PRS1_bar' in df.columns:
    normal_prs1 = df.loc[df['ATTACK_ID']==0,'p_PRS1_bar'].mean()
    print(f'\nPRS1 mean pressure (normal ops): {normal_prs1:.2f} barg (T4S: 18.0)')
if 'p_PRS2_bar' in df.columns:
    normal_prs2 = df.loc[df['ATTACK_ID']==0,'p_PRS2_bar'].mean()
    print(f'PRS2 mean pressure (normal ops): {normal_prs2:.2f} barg (T4S: 14.0)')

## 2. Exploratory Data Analysis

In [ ]:
# ── 2.1 Class Distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Attack type distribution
atk_counts = df['ATTACK_ID'].value_counts().sort_index()
atk_labels = {0:'Normal',1:'SrcPressure',2:'CompRatio',3:'ValveForce',
               4:'DemandManip',5:'PressSpoof',6:'FlowSpoof',
               7:'PLCLatency',8:'PipeLeak',9:'FDI',10:'Replay'}
axes[0].bar([atk_labels.get(i,str(i)) for i in atk_counts.index],
            atk_counts.values, color='steelblue', edgecolor='white')
axes[0].set_title('Attack Type Distribution')
axes[0].set_xlabel('Attack Type')
axes[0].set_ylabel('Row Count')
axes[0].tick_params(axis='x', rotation=45)

# Fault distribution
fault_labels = {0:'Normal',1:'PacketLoss',2:'StuckSensor'}
fault_counts = df['FAULT_ID'].value_counts().sort_index()
axes[1].pie(fault_counts.values,
            labels=[fault_labels.get(i,str(i)) for i in fault_counts.index],
            autopct='%1.1f%%', colors=['#2ecc71','#e74c3c','#f39c12'])
axes[1].set_title('Fault Type Distribution')

# Source config distribution
src_counts = df['source_config'].value_counts()
axes[2].bar(src_counts.index, src_counts.values, color='darkorange', edgecolor='white')
axes[2].set_title('Source Configuration Distribution')
axes[2].set_xlabel('Source Config')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'class_distribution.png', bbox_inches='tight')
plt.show()

print(f"Normal rows:  {(df['ATTACK_ID']==0).sum():,} ({100*(df['ATTACK_ID']==0).mean():.1f}%)")
print(f"Attack rows:  {(df['ATTACK_ID']>0).sum():,} ({100*(df['ATTACK_ID']>0).mean():.1f}%)")
print(f"Fault rows:   {(df['FAULT_ID']>0).sum():,} ({100*(df['FAULT_ID']>0).mean():.1f}%)")
print(f"Concurrent:   {((df['ATTACK_ID']>0)&(df['FAULT_ID']>0)).sum():,}")

In [ ]:
# ── 2.2 Normal vs Attack pressure profiles ──────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes = axes.flatten()

sample_size = min(5000, len(df))
df_sample = df.sample(sample_size, random_state=42)

plot_configs = [
    ('p_S1_bar',  'p_D1_bar',  'S1 vs D1 (CS1 PID loop)',   P_MIN_BARG, P_MAX_BARG),
    ('p_CS2_bar', 'p_PRS2_bar','CS2 outlet vs PRS2',        P_MIN_BARG, P_MAX_BARG),
    ('p_J7_bar',  'p_STO_bar', 'J7 (storage node) vs STO',  P_MIN_BARG, P_MAX_BARG),
    ('CS1_ratio', 'CS2_ratio', 'CS1 vs CS2 ratio',          1.0,        1.6),
]

for ax, (xcol, ycol, title, ymin, ymax) in zip(axes, plot_configs):
    if xcol not in df.columns or ycol not in df.columns:
        ax.set_title(f'{title} (N/A)')
        continue
    normal_m = df_sample['ATTACK_ID'] == 0
    ax.scatter(df_sample.loc[normal_m, xcol], df_sample.loc[normal_m, ycol],
               s=1, alpha=0.3, c='#2ecc71', label='Normal')
    ax.scatter(df_sample.loc[~normal_m, xcol], df_sample.loc[~normal_m, ycol],
               s=1, alpha=0.4, c='#e74c3c', label='Attack')
    ax.axhline(PRS1_SP, ls='--', c='orange', lw=0.8, label='PRS1 SP' if 'PRS' in ycol else '_')
    ax.axhline(PRS2_SP, ls=':', c='purple', lw=0.8, label='PRS2 SP' if 'PRS2' in ycol else '_')
    ax.set_xlabel(xcol); ax.set_ylabel(ycol)
    ax.set_title(title)
    ax.legend(markerscale=5, fontsize=8)

plt.suptitle('Indian CGD Pressure Operating Envelope (PNGRB T4S)', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pressure_scatter.png', bbox_inches='tight')
plt.show()

## 3. Feature Engineering (Physics-Derived Features)

In [ ]:
# ── 3.1 Weymouth Residuals (Tier 2 features) ────────────────────────────
# For each edge e: r_e = q_measured - q_Weymouth(P_i, P_j)
# Weymouth: Q = C * sqrt((P1^2 - P2^2) * D^(16/3) / (S*Z*T*L))

def weymouth_residual(df, edge, node_from, node_to, D_m, L_km,
                      S=0.57, Z=0.95, T_K=308):
    """Compute Weymouth predicted flow and residual for one edge."""
    p1_col = f'p_{node_from}_bar'
    p2_col = f'p_{node_to}_bar'
    q_col  = f'q_{edge}_scmd'
    if not all(c in df.columns for c in [p1_col, p2_col, q_col]):
        return pd.Series(np.nan, index=df.index), pd.Series(np.nan, index=df.index)

    p1 = df[p1_col].values
    p2 = df[p2_col].values
    C_w = 4.107e-3   # Weymouth constant (SI)
    L_m = L_km * 1e3

    dp2 = p1**2 - p2**2
    # Weymouth prediction (SCMD)
    q_pred = np.where(
        dp2 >= 0,
        C_w * np.sqrt(np.abs(dp2) * D_m**(16/3) / (S * Z * T_K * L_m)),
        -C_w * np.sqrt(np.abs(dp2) * D_m**(16/3) / (S * Z * T_K * L_m))
    )
    q_meas   = df[q_col].values
    residual = q_meas - q_pred
    return pd.Series(q_pred, index=df.index), pd.Series(residual, index=df.index)

# Compute residuals for key edges
edge_map = [
    ('E1','S1','J1',  0.202, 6),
    ('E3','CS1','J2', 0.202, 4),
    ('E7','CS2','J5', 0.202, 5),
    ('E12','J3','J7', 0.150,10),
    ('E16','J5','PRS2',0.150, 5),
]

print('Computing Weymouth residuals...')
for (edge, nf, nt, D, L) in edge_map:
    q_pred, resid = weymouth_residual(df, edge, nf, nt, D, L)
    df[f'weymouth_pred_{edge}']   = q_pred
    df[f'weymouth_resid_{edge}']  = resid
    print(f'  {edge} ({nf}→{nt}): resid σ={resid.std():.3f} SCMD')

WEYMOUTH_RESID_COLS = [c for c in df.columns if 'weymouth_resid' in c]
print(f'\n✓ Added {len(WEYMOUTH_RESID_COLS)} Weymouth residual columns')

In [ ]:
# ── 3.2 Kirchhoff Mass-Balance Residuals ────────────────────────────────
# At each junction: sum of inflows - sum of outflows should = 0

def kirchhoff_residual(df, node, in_edges, out_edges):
    """Net flow at a junction (should be ~0 in steady state)."""
    in_cols  = [f'q_{e}_scmd' for e in in_edges  if f'q_{e}_scmd' in df.columns]
    out_cols = [f'q_{e}_scmd' for e in out_edges if f'q_{e}_scmd' in df.columns]
    if not in_cols and not out_cols:
        return pd.Series(np.nan, index=df.index)
    in_sum  = df[in_cols].sum(axis=1)  if in_cols  else 0
    out_sum = df[out_cols].sum(axis=1) if out_cols else 0
    return in_sum - out_sum

# Junction mass balances (20-node network incidence matrix)
kirchhoff_map = [
    ('J2',  ['E3'],       ['E4','E8']),    # J2: CS1 in → trunk + side branch
    ('J3',  ['E4'],       ['E5','E12']),   # J3: trunk in → CS2 + upper
    ('J5',  ['E7','E15'], ['E16']),        # J5: CS2+storage → PRS2
    ('J7',  ['E12','E13'],['E14','E20']),  # J7: upper+S2 → storage+D6
]

print('Computing Kirchhoff mass-balance residuals...')
for (node, in_e, out_e) in kirchhoff_map:
    resid = kirchhoff_residual(df, node, in_e, out_e)
    df[f'kirchhoff_{node}'] = resid
    print(f'  {node}: σ={resid.std():.3f} SCMD | '
          f'max|r|={resid.abs().max():.2f} SCMD')

KIRCHHOFF_COLS = [c for c in df.columns if 'kirchhoff_' in c]

# ── 3.3 Rate-of-change features ─────────────────────────────────────────
print('\nComputing rate-of-change features...')
roc_targets = ['p_D1_bar','p_D3_bar','p_J7_bar','CS1_ratio','CS2_ratio']
for col in roc_targets:
    if col in df.columns:
        df[f'roc_{col}'] = df.groupby('scenario_id')[col].diff() * 10  # /dt (10 Hz)

ROC_COLS = [c for c in df.columns if c.startswith('roc_')]
print(f'  Added {len(ROC_COLS)} rate-of-change columns')

# ── 3.4 Compressor anomaly indicator ────────────────────────────────────
# CS1 over-compressing despite adequate D1 pressure → attack indicator
if 'CS1_ratio' in df.columns and 'p_D1_bar' in df.columns:
    df['cs1_anomaly'] = ((df['CS1_ratio'] >= 1.40) & (df['p_D1_bar'] > 17.0)).astype(int)
    print(f'  CS1 anomaly indicator: {df["cs1_anomaly"].sum():,} rows flagged')

print('\n✓ Feature engineering complete')
print(f'Total feature columns: {len(df.columns)}')

## 4. Train/Test Split (Scenario-Level Temporal)

In [ ]:
# ── 4.1 Scenario-level split (ICS_Dataset_Design.md §10.1) ──────────────
# IMPORTANT: Never split randomly across rows — keeps scenarios intact

all_scen_ids = sorted(df['scenario_id'].unique())
n_scenarios  = len(all_scen_ids)
print(f'Total scenarios: {n_scenarios}')

# Split 70/18/12
n_train = int(n_scenarios * 0.70)
n_val   = int(n_scenarios * 0.18)

train_ids = all_scen_ids[:n_train]
val_ids   = all_scen_ids[n_train:n_train+n_val]
test_ids  = all_scen_ids[n_train+n_val:]

df_train = df[df['scenario_id'].isin(train_ids)].copy()
df_val   = df[df['scenario_id'].isin(val_ids)].copy()
df_test  = df[df['scenario_id'].isin(test_ids)].copy()

print(f'Train: {len(df_train):,} rows ({len(train_ids)} scenarios)')
print(f'Val:   {len(df_val):,} rows ({len(val_ids)} scenarios)')
print(f'Test:  {len(df_test):,} rows ({len(test_ids)} scenarios)')

# Feature selection (Tier 1 + Tier 2)
FEATURE_COLS = (
    PRESSURE_COLS + FLOW_COLS[:10] +  # 20 P + 10 Q (most informative edges)
    EQUIP_COLS + EKF_RESID_COLS[:10] +
    WEYMOUTH_RESID_COLS + KIRCHHOFF_COLS + ROC_COLS +
    CUSUM_COLS + ['chi2_stat', 'jitter_ms']
)
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]
print(f'\nFeature matrix: {len(FEATURE_COLS)} features')

# Scale (fit on train only — no leakage from attack rows)
scaler = StandardScaler()
normal_train = df_train[df_train['ATTACK_ID']==0][FEATURE_COLS].fillna(0)
scaler.fit(normal_train)

X_train = scaler.transform(df_train[FEATURE_COLS].fillna(0))
X_val   = scaler.transform(df_val[FEATURE_COLS].fillna(0))
X_test  = scaler.transform(df_test[FEATURE_COLS].fillna(0))

y_train = df_train['ATTACK_ID'].values
y_val   = df_val['ATTACK_ID'].values
y_test  = df_test['ATTACK_ID'].values
y_bin_test = (y_test > 0).astype(int)

print('✓ Train/val/test matrices ready (scaler fitted on normal train data only)')

## 5. Unsupervised Anomaly Detection

In [ ]:
# ── 5.1 Isolation Forest ────────────────────────────────────────────────
print('Training Isolation Forest on normal train data...')
X_normal_train = X_train[y_train == 0]
contamination  = (y_test > 0).mean()  # use test attack fraction as prior

iso = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1
)
iso.fit(X_normal_train)

iso_scores  = iso.decision_function(X_test)   # higher = more normal
iso_preds   = (iso.predict(X_test) == -1).astype(int)  # -1 = anomaly

print('\nIsolation Forest Results (binary: normal vs any anomaly):')
print(classification_report(y_bin_test, iso_preds,
                             target_names=['Normal','Anomaly']))

# Per-attack-type detection rate
print('Detection rate by attack type:')
for aid in range(1, 11):
    mask = y_test == aid
    if mask.sum() == 0: continue
    dr = iso_preds[mask].mean() * 100
    atk_name = {1:'SrcPressure',2:'CompRatio',3:'Valve',4:'Demand',
                5:'PressSpoof',6:'FlowSpoof',7:'PLCLatency',
                8:'PipeLeak',9:'FDI',10:'Replay'}.get(aid,str(aid))
    print(f'  A{aid} {atk_name:12s}: {dr:5.1f}% ({mask.sum():5d} rows)')

In [ ]:
# ── 5.2 LSTM Autoencoder ─────────────────────────────────────────────────
WINDOW_SIZE = 30   # 30 steps = 30 s at 1 Hz
N_FEATURES  = len(FEATURE_COLS)

class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features, hidden_dim=64, latent_dim=16):
        super().__init__()
        self.encoder = nn.LSTM(n_features, hidden_dim, num_layers=2,
                               batch_first=True, dropout=0.2)
        self.enc_fc  = nn.Linear(hidden_dim, latent_dim)
        self.dec_fc  = nn.Linear(latent_dim, hidden_dim)
        self.decoder = nn.LSTM(hidden_dim, n_features, num_layers=2,
                               batch_first=True, dropout=0.2)

    def forward(self, x):
        out, (h, _) = self.encoder(x)
        z   = self.enc_fc(h[-1])           # latent vector
        z_  = self.dec_fc(z).unsqueeze(1).repeat(1, x.size(1), 1)
        rec, _ = self.decoder(z_)
        return rec

class WindowDataset(Dataset):
    def __init__(self, X, labels=None, window=30):
        self.X = torch.FloatTensor(X)
        self.y = labels
        self.w = window

    def __len__(self): return len(self.X) - self.w + 1

    def __getitem__(self, i):
        seq = self.X[i:i+self.w]
        if self.y is not None:
            return seq, int(self.y[i+self.w-1] > 0)
        return seq

# Train on normal data only (R05 equivalent)
X_normal_np = X_train[y_train == 0]
ds_train    = WindowDataset(X_normal_np, window=WINDOW_SIZE)
dl_train    = DataLoader(ds_train, batch_size=256, shuffle=True, num_workers=0)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training LSTM-AE on {device} ({len(ds_train):,} windows)...')

model     = LSTMAutoencoder(N_FEATURES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

train_losses = []
for epoch in range(20):
    model.train()
    epoch_loss = 0.0
    for batch_x, _ in dl_train if isinstance(ds_train[0], tuple) else [(b, None) for b in DataLoader(ds_train, 256, True)]:
        if isinstance(batch_x, (list,tuple)): batch_x = batch_x[0]
        batch_x = batch_x.to(device)
        rec = model(batch_x)
        loss = criterion(rec, batch_x)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(dl_train)
    train_losses.append(avg_loss)
    if (epoch+1) % 5 == 0:
        print(f'  Epoch {epoch+1:3d}/20  Loss: {avg_loss:.6f}')

print('\nLSTM-AE training complete')
torch.save(model.state_dict(), OUTPUT_DIR / 'lstm_ae.pt')

## 6. Supervised Classification

In [ ]:
# ── 6.1 XGBoost Multi-Class ──────────────────────────────────────────────
print('Training XGBoost classifier...')

# Class weights (inverse frequency)
class_counts = pd.Series(y_train).value_counts()
max_count    = class_counts.max()
class_weights = {cls: max_count/cnt for cls, cnt in class_counts.items()}
sample_weights = np.array([class_weights[y] for y in y_train])

xgb_model = xgb.XGBClassifier(
    n_estimators   = 500,
    max_depth      = 6,
    learning_rate  = 0.05,
    subsample      = 0.8,
    colsample_bytree = 0.8,
    eval_metric    = 'mlogloss',
    early_stopping_rounds = 20,
    random_state   = 42,
    n_jobs         = -1,
    verbosity      = 0
)

xgb_model.fit(
    X_train, y_train,
    sample_weight  = sample_weights,
    eval_set       = [(X_val, y_val)],
    verbose        = False
)

y_pred_xgb  = xgb_model.predict(X_test)
macro_f1    = f1_score(y_test, y_pred_xgb, average='macro')
print(f'\nXGBoost Results (macro F1 = {macro_f1:.4f}):')
target_names = ['Normal','A1-SrcP','A2-Comp','A3-Valve','A4-Dem',
                'A5-PSens','A6-Flow','A7-PLC','A8-Leak','A9-FDI','A10-Replay']
all_classes = sorted(np.unique(np.concatenate([y_test, y_pred_xgb])))
used_names  = [target_names[c] if c < len(target_names) else str(c) for c in all_classes]
print(classification_report(y_test, y_pred_xgb, labels=all_classes, target_names=used_names))

In [ ]:
# ── 6.2 Confusion Matrix ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(y_test, y_pred_xgb, labels=all_classes)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

im = ax.imshow(cm_pct, cmap='Blues')
ax.set_xticks(range(len(all_classes)))
ax.set_yticks(range(len(all_classes)))
ax.set_xticklabels(used_names, rotation=45, ha='right')
ax.set_yticklabels(used_names)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('XGBoost Confusion Matrix (%) — Indian CGD IDS')
plt.colorbar(im, ax=ax, label='% of true class')

for i in range(len(all_classes)):
    for j in range(len(all_classes)):
        txt = f'{cm_pct[i,j]:.0f}'
        ax.text(j, i, txt, ha='center', va='center',
                fontsize=8, color='white' if cm_pct[i,j]>50 else 'black')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'confusion_matrix_xgb.png', bbox_inches='tight')
plt.show()

## 7. SHAP Feature Attribution

In [ ]:
# ── 7.1 SHAP Analysis ────────────────────────────────────────────────────
print('Computing SHAP values (this may take 2–5 min)...')

# Use a sample for SHAP efficiency
shap_sample = min(2000, len(X_test))
X_shap = X_test[:shap_sample]
y_shap = y_test[:shap_sample]

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)

# Mean absolute SHAP per feature (averaged over all classes)
if isinstance(shap_values, list):
    shap_mean = np.mean([np.abs(sv).mean(axis=0) for sv in shap_values], axis=0)
else:
    shap_mean = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    'feature': FEATURE_COLS,
    'mean_abs_shap': shap_mean
}).sort_values('mean_abs_shap', ascending=False)

print('\nTop 20 features by mean |SHAP|:')
print(shap_df.head(20).to_string(index=False))

# Hypothesis H2 test: physics features vs raw sensor values
physics_cols = WEYMOUTH_RESID_COLS + KIRCHHOFF_COLS + ['chi2_stat','cusum_S_upper']
raw_p_cols   = [c for c in PRESSURE_COLS if c not in physics_cols]

shap_physics = shap_df[shap_df['feature'].isin(physics_cols)]['mean_abs_shap'].mean()
shap_raw_p   = shap_df[shap_df['feature'].isin(raw_p_cols)]['mean_abs_shap'].mean()

print(f'\n=== Hypothesis H2 Test ===')
print(f'Mean |SHAP| physics features: {shap_physics:.5f}')
print(f'Mean |SHAP| raw pressure:     {shap_raw_p:.5f}')
if shap_physics > shap_raw_p:
    print('✓ H2 SUPPORTED: physics-derived features rank above raw sensor values')
else:
    print('✗ H2 NOT SUPPORTED for this model configuration')

In [ ]:
# ── 7.2 SHAP Beeswarm Plot ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

top_n   = 20
top_idx = shap_df.head(top_n)['feature'].tolist()
feat_idx = [FEATURE_COLS.index(f) for f in top_idx if f in FEATURE_COLS]

if isinstance(shap_values, list):
    sv_plot = np.mean([np.abs(sv[:,feat_idx]) for sv in shap_values], axis=0)
else:
    sv_plot = shap_values[:,feat_idx]

sv_mean  = np.abs(sv_plot).mean(axis=0)
sorted_i = np.argsort(sv_mean)[::-1]

bars = ax.barh(range(len(sorted_i)),
               sv_mean[sorted_i],
               color=['#e74c3c' if 'weymouth' in top_idx[i] or
                       'kirchhoff' in top_idx[i] or 'chi2' in top_idx[i]
                       else '#3498db' for i in sorted_i])

ax.set_yticks(range(len(sorted_i)))
ax.set_yticklabels([top_idx[i] for i in sorted_i], fontsize=8)
ax.set_xlabel('Mean |SHAP Value|')
ax.set_title('Top 20 Features by SHAP Importance\n'
             'Red = physics-derived, Blue = raw sensor')
ax.invert_yaxis()

# Add legend patches
import matplotlib.patches as mpatches
ax.legend(handles=[
    mpatches.Patch(color='#e74c3c', label='Physics-derived (Weymouth/Kirchhoff/EKF)'),
    mpatches.Patch(color='#3498db', label='Raw sensor values')
], fontsize=9)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'shap_importance.png', bbox_inches='tight')
plt.show()

## 8. Cross-Topology Validation

In [ ]:
# ── 8.1 Leave-One-Topology-Out CV ────────────────────────────────────────
# Group scenarios by source_config × cs_mode (9 topology groups)
df['topology_group'] = df['source_config'] + '_' + df['cs_mode']
topology_groups = df['topology_group'].unique()
print(f'Topology groups ({len(topology_groups)}): {sorted(topology_groups)}')

cv_results = []

for holdout in topology_groups:
    mask_test  = df['topology_group'] == holdout
    mask_train = ~mask_test

    X_cv_train = scaler.transform(df.loc[mask_train, FEATURE_COLS].fillna(0))
    X_cv_test  = scaler.transform(df.loc[mask_test,  FEATURE_COLS].fillna(0))
    y_cv_train = df.loc[mask_train, 'ATTACK_ID'].values
    y_cv_test  = df.loc[mask_test,  'ATTACK_ID'].values

    if len(np.unique(y_cv_train)) < 2: continue

    sw = np.array([class_weights.get(y, 1.0) for y in y_cv_train])
    model_cv = xgb.XGBClassifier(
        n_estimators=200, max_depth=5, learning_rate=0.1,
        random_state=42, n_jobs=-1, verbosity=0
    )
    model_cv.fit(X_cv_train, y_cv_train, sample_weight=sw)
    y_cv_pred = model_cv.predict(X_cv_test)

    f1 = f1_score(y_cv_test, y_cv_pred, average='macro', zero_division=0)
    acc = (y_cv_pred == y_cv_test).mean()
    cv_results.append({'topology': holdout, 'macro_f1': f1, 'accuracy': acc,
                       'n_test': mask_test.sum()})
    print(f'  Holdout {holdout:25s}: F1={f1:.4f}  Acc={acc:.4f}  N={mask_test.sum():6,}')

cv_df = pd.DataFrame(cv_results)
print(f'\nMean macro F1 across topologies: {cv_df["macro_f1"].mean():.4f} ± {cv_df["macro_f1"].std():.4f}')
print(f'Min macro F1: {cv_df["macro_f1"].min():.4f} (topology: {cv_df.loc[cv_df["macro_f1"].idxmin(),"topology"]})')
cv_df.to_csv(OUTPUT_DIR / 'cross_topology_cv_results.csv', index=False)

In [ ]:
# ── 8.2 CV Results Plot ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
cv_sorted = cv_df.sort_values('macro_f1')
bars = ax.barh(cv_sorted['topology'], cv_sorted['macro_f1'],
               color=['#e74c3c' if v < 0.7 else '#27ae60' for v in cv_sorted['macro_f1']])
ax.axvline(0.7, ls='--', c='black', alpha=0.5, label='F1=0.70 threshold')
ax.axvline(cv_df['macro_f1'].mean(), ls='-.', c='blue', alpha=0.7,
           label=f'Mean F1={cv_df["macro_f1"].mean():.3f}')
ax.set_xlabel('Macro F1 Score')
ax.set_title('Leave-One-Topology-Out Cross-Validation\n'
             'Indian CGD Network — 9 Topology Groups')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cross_topology_cv.png', bbox_inches='tight')
plt.show()

## 9. Resilience Scenario Analysis

In [ ]:
# ── 9.1 Resilience vs Normal Scenario Comparison ─────────────────────────
if 'scenario_resilience_mode' in df.columns:
    res_modes = df['scenario_resilience_mode'].value_counts()
    print('Resilience mode distribution:')
    print(res_modes.to_string())

    # Compare pressure at D1/D3 in normal vs crosstie scenarios
    normal_mode = df[df['scenario_resilience_mode']=='normal']
    crosstie_mode = df[df['scenario_resilience_mode']=='crosstie']

    if len(crosstie_mode) > 0 and 'p_D3_bar' in df.columns:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        for ax, col, title in zip(axes,
                                   ['p_D1_bar','p_D3_bar'],
                                   ['D1 Supply Pressure','D3 Supply Pressure']):
            normal_vals   = normal_mode[col].dropna()
            crosstie_vals = crosstie_mode[col].dropna()

            ax.hist(normal_vals, bins=50, alpha=0.6, color='#3498db',
                    label=f'Normal (μ={normal_vals.mean():.1f} barg)', density=True)
            ax.hist(crosstie_vals, bins=50, alpha=0.6, color='#e67e22',
                    label=f'Cross-tie E21 (μ={crosstie_vals.mean():.1f} barg)', density=True)
            ax.axvline(PRS1_SP if 'D1' in col else PRS2_SP,
                       ls='--', c='red', label='PNGRB T4S setpoint')
            ax.set_xlabel('Pressure (barg)')
            ax.set_ylabel('Density')
            ax.set_title(f'{title}\nNormal vs E21 Cross-Tie Active')
            ax.legend(fontsize=8)

        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / 'resilience_pressure_comparison.png', bbox_inches='tight')
        plt.show()

        # Quantify supply continuity improvement
        for col, sp, demand_node in [('p_D1_bar', PRS1_SP, 'D1'),
                                       ('p_D3_bar', PRS2_SP, 'D3')]:
            if col not in df.columns: continue
            n_starved_normal   = (normal_mode[col] < sp*0.9).mean()
            n_starved_crosstie = (crosstie_mode[col] < sp*0.9).mean()
            print(f'{demand_node} under-supply (<90% setpoint):')
            print(f'  Normal mode:    {n_starved_normal*100:.1f}%')
            print(f'  Cross-tie mode: {n_starved_crosstie*100:.1f}%')
            print(f'  Improvement: {(n_starved_normal - n_starved_crosstie)*100:.1f} percentage points')
else:
    print('No resilience mode data found (run resilience scenarios first)')

## 10. Export Results Summary

In [ ]:
# ── 10.1 Comprehensive Results Table ────────────────────────────────────
results_table = pd.DataFrame({
    'Model':       ['Isolation Forest', 'LSTM-AE', 'XGBoost', 'XGBoost (CV avg)'],
    'Type':        ['Unsupervised','Unsupervised','Supervised','Supervised'],
    'Macro_F1':    [
        f1_score(y_bin_test, iso_preds, average='macro'),
        np.nan,   # fill after LSTM-AE evaluation
        f1_score(y_test, y_pred_xgb, average='macro'),
        cv_df['macro_f1'].mean() if 'cv_df' in dir() else np.nan
    ],
    'Notes': [
        'Normal-only training, PNGRB T4S contamination prior',
        'LSTM-AE reconstruction error threshold',
        'Scale-weighted, 500 estimators, Tier1+Tier2 features',
        'Leave-one-topology-out, 9 folds'
    ]
})
print(results_table.to_string(index=False))
results_table.to_csv(OUTPUT_DIR / 'model_comparison.csv', index=False)

# ── 10.2 Dataset statistics for paper ────────────────────────────────────
dataset_stats = {
    'total_rows':        len(df),
    'n_scenarios':       df['scenario_id'].nunique(),
    'n_features':        len(FEATURE_COLS),
    'attack_classes':    10,
    'normal_fraction':   float((df['ATTACK_ID']==0).mean()),
    'attack_fraction':   float((df['ATTACK_ID']>0).mean()),
    'fault_fraction':    float((df['FAULT_ID']>0).mean()),
    'pressure_range_barg': [float(df[PRESSURE_COLS].min().min()),
                             float(df[PRESSURE_COLS].max().max())],
    'regulatory_basis':  'PNGRB_T4S_2024',
    'gas_sg':            0.57,
    'topology':          '20-node CGD steel grid + 2 resilience edges',
    'sampling_rate_hz':  1,
    'physics_rate_hz':   10,
}

with open(OUTPUT_DIR / 'dataset_statistics.json', 'w') as f:
    json.dump(dataset_stats, f, indent=2)
print('\n✓ Dataset statistics saved to ml_outputs/dataset_statistics.json')

print('\n=== Pipeline Complete ===')
print(f'Outputs in: {OUTPUT_DIR.absolute()}')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name}')